# 🔑 Comprehensive RSA Encryption and Decryption Implementation

This notebook provides a complete Python implementation of the RSA cryptographic algorithm. We'll cover all the foundational mathematical helper functions, implement the core `RSAUser` class for key generation, encryption, and decryption, and then demonstrate its practical use through several examples, including an interactive messenger.

## 1. Essential Mathematical Helper Functions for RSA

RSA relies heavily on number theory. These functions provide the mathematical building blocks required for key generation and cryptographic operations.

*   **`gcd(a, b)`**: The Greatest Common Divisor function from Python's `math` module, crucial for ensuring 'e' and 'phi(n)' are coprime.
*   **`egcd(a, b)`**: The Extended Euclidean Algorithm. This is vital for finding the modular multiplicative inverse, which is needed to calculate the private key 'd'. It computes `g, x, y` such that `ax + by = g = gcd(a, b)`.
*   **`is_prime(num)`**: A utility function to check if a given number is prime. This is used during key generation to ensure that 'p' and 'q' are indeed prime numbers.
*   **`mod_inverse(a, m)`**: Calculates the modular multiplicative inverse of `a` modulo `m`. This means finding a number `x` such that `(a * x) % m = 1`. This function is now a static method within the `RSAUser` class.
*   **`mod_exp(base, exponent, modulus)`**: Performs fast modular exponentiation (also known as repeated squaring). This efficiently calculates `(base^exponent) % modulus`, which is fundamental for both encryption and decryption operations in RSA. This function is also now a static method within the `RSAUser` class.

In [16]:
import math # Ensure math is imported for gcd

# === Helper Function: is_prime ===
# This function checks if a number is prime. Prime numbers p and q are the foundation of RSA security.
def is_prime(num: int) -> bool:
    """Checks if a number is prime.

    Args:
        num (int): The number to check.

    Returns:
        bool: True if the number is prime, False otherwise.
    """
    if num < 2:
        return False
    for i in range(2, int(num**0.5) + 1):
        if num % i == 0:
            return False
    return True

# === Helper Function: egcd (Extended Euclidean Algorithm) ===
# Used to find the modular inverse, a critical step in deriving the private key 'd'.
def egcd(a: int, b: int) -> tuple[int, int, int]:
    """Extended Euclidean Algorithm.

    Computes g, x, y such that ax + by = g = gcd(a, b).

    Args:
        a (int): First integer.
        b (int): Second integer.

    Returns:
        tuple[int, int, int]: A tuple (g, x, y) where g is the GCD, and x, y are coefficients.
    """
    if b == 0:
        return a, 1, 0
    g, x1, y1 = egcd(b, a % b)
    return g, y1, x1 - (a // b) * y1

## 2. The `RSAUser` Class: Encapsulating RSA Logic

This class represents an individual participant in an RSA communication system. It handles all aspects of RSA for a single user, from key generation to message encryption and decryption. This object-oriented approach ensures that each user has their own distinct keys and can manage their cryptographic operations independently.

*   **`__init__(self, name, p, q, e)`**: The constructor for an `RSAUser`. It takes the user's name, two large prime numbers `p` and `q`, and a public exponent `e`. It then calculates and stores `n` (modulus), `phi` (Euler's totient function), and `d` (private exponent). Importantly, it includes validation to ensure `p`, `q` are distinct primes and `e` is valid.
*   **`explain_key_generation()`**: A method to print a detailed, step-by-step breakdown of how the user's RSA public and private keys were derived. This is excellent for understanding the underlying mathematics.
*   **`encrypt_number(self, m, receiver_public_key)`**: Encrypts a single integer message `m` intended for another user, using that receiver's public key `(n, e)`. The result is the ciphertext `c`.
*   **`decrypt_number(self, c)`**: Decrypts an integer ciphertext `c` using the user's own private key `(n, d)`, recovering the original numerical message `m`.
*   **`encrypt_text(self, message, receiver_public_key)`**: A higher-level function to encrypt an entire string message. It converts each character into its ASCII value (an integer) and encrypts each number individually using the receiver's public key.
*   **`decrypt_text(self, ciphertext)`**: The inverse of `encrypt_text`. It takes a list of encrypted integers and decrypts each one, then converts the recovered ASCII values back into a readable string message.

In [17]:
class RSAUser:
    """Represents an individual participant in an RSA communication system.

    Handles all aspects of RSA for a single user, from key generation to message encryption and decryption.
    """

    # === Static Method: mod_inverse (Modular Multiplicative Inverse) ===
    # This method is crucial for calculating the private key 'd'.
    # It finds 'd' such that (e * d) % phi(n) = 1.
    @staticmethod
    def mod_inverse(a: int, m: int) -> int:
        """Find d such that a*d ≡ 1 (mod m) using the Extended Euclidean Algorithm.

        Args:
            a (int): The number for which to find the modular inverse.
            m (int): The modulus.

        Returns:
            int: The modular multiplicative inverse d.

        Raises:
            ValueError: If the modular inverse does not exist (a and m are not coprime).
        """
        g, x, _ = egcd(a, m)
        if g != 1:
            raise ValueError("Modular inverse does not exist. 'a' and 'm' must be coprime.")
        return x % m

    # === Static Method: mod_exp (Modular Exponentiation / Repeated Squaring) ===
    # This is the core mathematical operation for both RSA encryption and decryption.
    # It efficiently calculates (base^exponent) % modulus.
    @staticmethod
    def mod_exp(base: int, exponent: int, modulus: int, show_steps: bool = False) -> int:
        """
        Fast modular exponentiation using repeated squaring.
        This efficiently calculates (base^exponent) % modulus.

        Args:
            base (int): The base number.
            exponent (int): The exponent.
            modulus (int): The modulus.
            show_steps (bool, optional): If True, print detailed steps of the calculation. Defaults to False.

        Returns:
            int: The result of (base^exponent) % modulus.
        """
        result = 1
        base %= modulus
        step = 1

        if show_steps:
            print(f"\n=== GREEN MINI TITLE: REPEATED SQUARING STEPS ===")
            print(f"Repeated squaring for ({base}^{exponent}) mod {modulus}")

        while exponent > 0:
            if show_steps:
                print(
                    f"Step {step}: result={result}, base={base}, exponent={exponent}"
                )
            # If exponent is odd, multiply result by current base
            if exponent % 2 == 1:
                result = (result * base) % modulus
                if show_steps:
                    print(f"  exponent is odd -> result = (result * base) mod modulus = {result}")

            # Square the base for the next iteration
            base = (base * base) % modulus
            exponent //= 2 # Halve the exponent
            step += 1

        if show_steps:
            print(f"=== END REPEATED SQUARING STEPS ===\n")
        return result

    # === RSAUser Constructor: __init__ ===
    # Initializes an RSA user, generating all necessary public and private keys.
    def __init__(self, name: str, p: int, q: int, e: int):
        """Initializes an RSAUser instance, generating public and private keys.

        Args:
            name (str): The name of the user.
            p (int): A large prime number.
            q (int): Another large, distinct prime number.
            e (int): The public exponent.

        Raises:
            ValueError: If p or q are not distinct primes, or if e is not coprime with phi(n).
        """
        self.name = name

        # Key Generation Validation: p and q must be distinct prime numbers
        if not (is_prime(p) and is_prime(q)):
            raise ValueError(f"Error for {name}: p ({p}) and q ({q}) must both be prime numbers.")
        if p == q:
            raise ValueError(f"Error for {name}: p ({p}) and q ({q}) must be distinct prime numbers.")

        self.p = p
        self.q = q
        self.n = p * q  # Modulus: n = p * q
        self.phi = (p - 1) * (q - 1) # Euler's Totient Function: phi(n) = (p-1)(q-1)
        self.e = e # Public exponent

        # Key Generation Validation: e must be coprime with phi(n)
        if math.gcd(self.e, self.phi) != 1:
            raise ValueError(
                f"Invalid public exponent 'e' for {name}: gcd(e, phi(n)) must be 1. "
                f"Got gcd({self.e}, {self.phi}) = {math.gcd(self.e, self.phi)} != 1"
                f" Please choose a different 'e' or check p and q."
            )

        # Calculate private exponent 'd' using modular inverse
        self.d = self.mod_inverse(self.e, self.phi)
        self.public_key = (self.n, self.e)
        self.private_key = (self.n, self.d)

    # === Method: explain_key_generation ===
    # Provides a transparent step-by-step explanation of how the RSA keys were formed.
    def explain_key_generation(self):
        """Prints a detailed, step-by-step breakdown of how the user's RSA keys were derived.
        """
        print(f"\n=== {self.name}'s RSA Key Generation Details ===")
        print(f"1. Choose two distinct prime numbers: p = {self.p}, q = {self.q}")
        print(f"2. Compute the modulus: n = p * q = {self.p} * {self.q} = {self.n}")
        print(f"3. Compute Euler's totient function: phi(n) = (p - 1)(q - 1) = ({self.p} - 1) * ({self.q} - 1) = {self.phi}")
        print(f"4. Choose a public exponent 'e' such that 1 < e < phi(n) and gcd(e, phi(n)) = 1")
        print(f"   Chosen e = {self.e}")
        print(f"5. Compute the private exponent 'd' such that d * e ≡ 1 mod phi(n)")
        print(f"   So d * {self.e} ≡ 1 mod {self.phi}")
        print(f"   Calculated d = {self.d}")
        print(f"6. Public Key (for sharing): (n, e) = {self.public_key}")
        print(f"7. Private Key (keep secret!): (n, d) = {self.private_key}")
        print(f"========================================")

    # === Method: encrypt_number ===
    # Encrypts a single numerical message using the receiver's public key.
    def encrypt_number(self, m: int, receiver_public_key: tuple[int, int], show_steps: bool = False) -> int:
        """Encrypts a single integer message using the receiver's public key.

        Args:
            m (int): The integer message to encrypt.
            receiver_public_key (tuple[int, int]): The receiver's public key (n, e).
            show_steps (bool, optional): If True, print detailed encryption steps. Defaults to False.

        Returns:
            int: The encrypted ciphertext c.

        Raises:
            ValueError: If the message m is not within the valid range (0 <= m < n).
        """
        n, e = receiver_public_key
        if not (0 <= m < n):
            raise ValueError(f"Message number 'm' ({m}) must satisfy 0 <= m < n ({n}) for direct encryption.")
        if show_steps:
            print(f"\n=== GREEN MINI TITLE: ENCRYPTING NUMBER ===")
            print(f"Encrypting message m = {m} using receiver's public key (n={n}, e={e})")
            print(f"Calculation: c = m^e mod n = {m}^{e} mod {n}")
        c = self.mod_exp(m, e, n, show_steps) # Use the fast modular exponentiation
        if show_steps:
            print(f"Resulting Ciphertext c = {c}")
            print(f"=== END ENCRYPTING NUMBER ===\n")
        return c

    # === Method: decrypt_number ===
    # Decrypts a single numerical ciphertext using the user's own private key.
    def decrypt_number(self, c: int, show_steps: bool = False) -> int:
        """Decrypts a single integer ciphertext using the user's private key.

        Args:
            c (int): The integer ciphertext to decrypt.
            show_steps (bool, optional): If True, print detailed decryption steps. Defaults to False.

        Returns:
            int: The recovered original integer message m.
        """
        n, d = self.private_key
        if show_steps:
            print(f"\n=== GREEN MINI TITLE: DECRYPTING NUMBER ===")
            print(f"Decrypting ciphertext c = {c} using own private key (n={n}, d={d})")
            print(f"Calculation: m = c^d mod n = {c}^{d} mod {n}")
        m = self.mod_exp(c, d, n, show_steps) # Use the fast modular exponentiation
        if show_steps:
            print(f"Recovered Message m = {m}")
            print(f"=== END DECRYPTING NUMBER ===\n")
        return m

    # === Method: encrypt_text ===
    # Converts a string into a list of encrypted numbers.
    def encrypt_text(self, message: str, receiver_public_key: tuple[int, int]) -> list[int]:
        """Encrypts a string message by converting each character to ASCII and encrypting individually.

        Args:
            message (str): The string message to encrypt.
            receiver_public_key (tuple[int, int]): The receiver's public key (n, e).

        Returns:
            list[int]: A list of encrypted integers (ciphertext).

        Raises:
            ValueError: If a character's ASCII value is too large for the modulus n.
        """
        n, _ = receiver_public_key
        encrypted = []
        print(f"\n=== GREEN MINI TITLE: ENCRYPTING TEXT '{message}' ===")
        print(f"Converting each character to ASCII and encrypting individually...")
        for ch in message:
            value = ord(ch) # Get ASCII value of character
            if value >= n:
                raise ValueError(
                    f"Character '{ch}' has ASCII code {value}, which is too large for modulus n = {n}. "
                    "Choose larger primes (p, q) or implement block encoding for longer messages/larger character sets."
                )
            print(f"  Encrypting '{ch}' (ASCII: {value})")
            encrypted.append(self.encrypt_number(value, receiver_public_key))
        print(f"Text encryption complete. Ciphertext list: {encrypted}")
        print(f"=== END ENCRYPTING TEXT ===\n")
        return encrypted

    # === Method: decrypt_text ===
    # Converts a list of encrypted numbers back into a readable string.
    def decrypt_text(self, ciphertext: list[int]) -> str:
        """Decrypts a list of encrypted integers back into a readable string message.

        Args:
            ciphertext (list[int]): A list of encrypted integers.

        Returns:
            str: The recovered original string message.
        """
        print(f"\n=== GREEN MINI TITLE: DECRYPTING TEXT ===")
        print(f"Decrypting each number and converting back to characters...")
        decrypted_chars = []
        for c in ciphertext:
            decrypted_val = self.decrypt_number(c)
            decrypted_chars.append(chr(decrypted_val)) # Convert ASCII back to character
            print(f"  Decrypted {c} to {decrypted_val} (char: {chr(decrypted_val)}) ")
        recovered_message = "".join(decrypted_chars)
        print(f"Text decryption complete. Recovered message: '{recovered_message}'")
        print(f"=== END DECRYPTING TEXT ===\n")
        return recovered_message

## 3. RSA Demonstrations: Putting it All Together

This section showcases the RSA algorithm in a practical (though simplified) scenario. It helps visualize the key generation, encryption, and decryption processes through an interactive messenger.

*   **`interactive_messenger()`**: Provides a command-line interface where you can define two custom RSA users, generate their keys, and then interactively send numerical messages between them. This allows for hands-on experimentation with different prime numbers and messages.

In [18]:
def interactive_messenger():
    """Provides an interactive command-line interface for RSA message exchange.

    Allows two users to be defined, generates their RSA keys, and facilitates sending
    numerical messages between them, demonstrating encryption and decryption steps.
    """
    print("\n" + "=" * 60)
    print("=== GREEN MINI TITLE: INTERACTIVE RSA MESSENGER ===")
    print("   Define your own users and exchange numerical messages.")
    print("   (Remember to use distinct small prime numbers for p and q for learning purposes!)")
    print("=" * 60)

    # --- User 1 Setup ---
    user1_name = input("Enter first user's name: ")
    while True:
        try:
            p1 = int(input(f"Enter {user1_name}'s prime p: "))
            q1 = int(input(f"Enter {user1_name}'s prime q: "))
            if not (is_prime(p1) and is_prime(q1)):
                raise ValueError("Both numbers must be prime.")
            if p1 == q1:
                raise ValueError("Primes p and q must be distinct.")
            break
        except ValueError as e:
            print(f"Invalid input for primes: {e}. Please try again.")

    phi1 = (p1 - 1) * (q1 - 1)
    e1_candidate = 3
    while math.gcd(e1_candidate, phi1) != 1:
        e1_candidate += 2  # Increment by 2 to keep it odd (or 1 if 2 is coprime)
    e1 = e1_candidate

    print("\n--- Initializing RSA Users with generated keys ---")
    user1 = RSAUser(user1_name, p1, q1, e1)
    print(f"Automatically chosen public exponent e for {user1.name}: {e1}")

    # --- User 2 Setup ---
    user2_name = input("Enter second user's name: ")
    while True:
        try:
            p2 = int(input(f"Enter {user2_name}'s prime p: "))
            q2 = int(input(f"Enter {user2_name}'s prime q: "))
            if not (is_prime(p2) and is_prime(q2)):
                raise ValueError("Both numbers must be prime.")
            if p2 == q2:
                raise ValueError("Primes p and q must be distinct.")
            break
        except ValueError as e:
            print(f"Invalid input for primes: {e}. Please try again.")

    phi2 = (p2 - 1) * (q2 - 1)
    e2_candidate = 3
    while math.gcd(e2_candidate, phi2) != 1:
        e2_candidate += 2
    e2 = e2_candidate

    user2 = RSAUser(user2_name, p2, q2, e2)
    print(f"Automatically chosen public exponent e for {user2.name}: {e2}")

    user1.explain_key_generation()
    user2.explain_key_generation()

    while True:
        print("\n" + "=" * 60)
        print("=== GREEN MINI TITLE: CHOOSE A MESSAGING OPTION ===")
        print("1. Send a number from User 1 to User 2")
        print("2. Send a number from User 2 to User 1")
        print("3. Quit Interactive Messenger")
        choice = input("Your choice: ").strip()

        if choice == "1":
            receiver_n = user2.public_key[0]
            max_m = receiver_n - 1
            while True:
                try:
                    message_to_send = int(input(f"Enter a number message for {user1.name} to send to {user2.name} (max {max_m}): "))
                    if not (0 <= message_to_send < receiver_n):
                        raise ValueError(f"Message number 'm' ({message_to_send}) must satisfy 0 <= m < n ({receiver_n}) for direct encryption.")
                    break
                except ValueError as e:
                    print(f"Error during message exchange: {e}. Please try again.")

            print(f"\n--- {user1.name} encrypts for {user2.name} ---")
            ciphertext = user1.encrypt_number(message_to_send, user2.public_key, show_steps=True)
            print(f"Ciphertext: {ciphertext}")

            print(f"\n--- {user2.name} decrypts message from {user1.name} ---")
            decrypted_message = user2.decrypt_number(ciphertext, show_steps=True)
            print(f"Final decrypted message for {user2.name}: {decrypted_message} (Original: {message_to_send})")
            if decrypted_message == message_to_send:
                print(f"Message successfully transmitted from {user1.name} to {user2.name}!")
            else:
                print(f"ERROR: Decrypted message does not match original!")

        elif choice == "2":
            receiver_n = user1.public_key[0]
            max_m = receiver_n - 1
            while True:
                try:
                    message_to_send = int(input(f"Enter a number message for {user2.name} to send to {user1.name} (max {max_m}): "))
                    if not (0 <= message_to_send < receiver_n):
                        raise ValueError(f"Message number 'm' ({message_to_send}) must satisfy 0 <= m < n ({receiver_n}) for direct encryption.")
                    break
                except ValueError as e:
                    print(f"Error during message exchange: {e}. Please try again.")

            print(f"\n--- {user2.name} encrypts for {user1.name} ---")
            ciphertext = user2.encrypt_number(message_to_send, user1.public_key, show_steps=True)
            print(f"Ciphertext: {ciphertext}")

            print(f"\n--- {user1.name} decrypts message from {user2.name} ---")
            decrypted_message = user1.decrypt_number(ciphertext, show_steps=True)
            print(f"Final decrypted message for {user1.name}: {decrypted_message} (Original: {message_to_send})")
            if decrypted_message == message_to_send:
                print(f"Message successfully transmitted from {user2.name} to {user1.name}!")
            else:
                print(f"ERROR: Decrypted message does not match original!")

        elif choice == "3":
            print("Exiting Interactive Messenger. Goodbye!")
            break
        else:
            print("Invalid choice. Please enter 1, 2, or 3.")

## 4. Run the Demonstrations

Execute the cell below to run the predefined RSA demonstrations and, finally, launch the interactive messenger. This will allow you to see the RSA algorithm in action and experiment with your own inputs.

In [ ]:
if __name__ == "__main__":
    print("\n--- Running RSA Demonstrations ---")


    interactive_messenger()


--- Running RSA Demonstrations ---

=== GREEN MINI TITLE: INTERACTIVE RSA MESSENGER ===
   Define your own users and exchange numerical messages.
   (Remember to use distinct small prime numbers for p and q for learning purposes!)
Enter first user's name: Youssef
Enter Youssef's prime p: 89
Enter Youssef's prime q: 67

--- Initializing RSA Users with generated keys ---
Automatically chosen public exponent e for Youssef: 5
Enter second user's name: Prof  Nacho
Enter Prof  Nacho's prime p: 61
Enter Prof  Nacho's prime q: 59
Automatically chosen public exponent e for Prof  Nacho: 7

=== Youssef's RSA Key Generation Details ===
1. Choose two distinct prime numbers: p = 89, q = 67
2. Compute the modulus: n = p * q = 89 * 67 = 5963
3. Compute Euler's totient function: phi(n) = (p - 1)(q - 1) = (89 - 1) * (67 - 1) = 5808
4. Choose a public exponent 'e' such that 1 < e < phi(n) and gcd(e, phi(n)) = 1
   Chosen e = 5
5. Compute the private exponent 'd' such that d * e ≡ 1 mod phi(n)
   So d * 